DATA LOAD

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('student_data.csv')
print(df.shape)
df.info()
print(df.isnull().sum().sum())
df.head()


(395, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


DATA PREP

In [3]:
df = df.drop(['G1', 'G2'], axis=1)               # remove columns that leak G3
df = pd.get_dummies(df, drop_first=True, dtype=int)   # text -> 0/1 columns

def check_pass(score):
    if score >= 10:
        return 1
    else:
        return 0

df['pass'] = df['G3'].apply(check_pass)
print(df['pass'].value_counts())                 # 265 pass, 130 fail

X = df.drop(['G3', 'pass'], axis=1)              # clues
y_reg = df['G3']                                 # answer for regression
y_class = df['pass']                             # answer for classification
print(X.shape, y_reg.shape, y_class.shape)       # (395, 39) (395,) (395,)

pass
1    265
0    130
Name: count, dtype: int64
(395, 39) (395,) (395,)


TRAIN TEST SPLIT

In [4]:
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class)

print(X_train_reg.shape, X_test_reg.shape)       # (316, 39) (79, 39)

(316, 39) (79, 39)


PREDICT SCORE

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

model_reg = LinearRegression()
model_reg.fit(X_train_reg, y_train_reg)
preds_reg = model_reg.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds_reg))

baseline = [y_train_reg.mean()] * len(y_test_reg)
print("Baseline MAE:", mean_absolute_error(y_test_reg, baseline))

MAE: 3.39526092580192
Baseline MAE: 3.64585002403461


PREDICT PASS FAIL

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

model_class = LogisticRegression(max_iter=1000)
model_class.fit(X_train_class, y_train_class)
preds_class = model_class.predict(X_test_class)

print("Accuracy:", accuracy_score(y_test_class, preds_class))
print(confusion_matrix(y_test_class, preds_class))
print("Baseline accuracy:", y_test_class.mean())

Accuracy: 0.6582278481012658
[[ 9 17]
 [10 43]]
Baseline accuracy: 0.6708860759493671


BETTER EVALUATION

In [10]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
print("Logistic Regression:", cross_val_score(lr, X, y_class, cv=5).mean())

lr_bal = LogisticRegression(max_iter=1000, class_weight='balanced')
print(classification_report(y_class, cross_val_predict(lr_bal, X, y_class, cv=5)))

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
print(classification_report(y_class, cross_val_predict(rf, X, y_class, cv=5)))

df.head()
df[['G3', 'pass']].head(10)

Logistic Regression: 0.6860759493670886
              precision    recall  f1-score   support

           0       0.46      0.52      0.49       130
           1       0.75      0.70      0.73       265

    accuracy                           0.64       395
   macro avg       0.61      0.61      0.61       395
weighted avg       0.66      0.64      0.65       395

              precision    recall  f1-score   support

           0       0.61      0.24      0.34       130
           1       0.71      0.92      0.80       265

    accuracy                           0.70       395
   macro avg       0.66      0.58      0.57       395
weighted avg       0.68      0.70      0.65       395



,G3,pass
0,6,0
1,6,0
2,10,1
3,15,1
4,10,1
5,15,1
6,11,1
7,6,0
8,19,1
9,15,1


   age  Medu  Fedu  traveltime  studytime  failures  famrel  freetime  goout  \
0   18     4     4           2          2         0       4         3      4   
1   17     1     1           1          2         0       5         3      3   
2   15     1     1           1          2         3       4         3      2   
3   15     4     2           1          3         0       3         2      2   
4   16     3     3           1          2         0       4         3      2   

   Dalc  ...  guardian_other  schoolsup_yes  famsup_yes  paid_yes  \
0     1  ...               0              1           0         0   
1     1  ...               0              0           1         0   
2     2  ...               0              1           0         1   
3     1  ...               0              0           1         1   
4     1  ...               0              0           1         1   

   activities_yes  nursery_yes  higher_yes  internet_yes  romantic_yes  pass  
0               0        